# Exploratory Data Analysis (EDA) & Data Cleaning
### Internship Task: Week 1 – Task 3
**Author:** Sayantani Adhikary  
**Objective:** Perform end-to-end data profiling, missing value imputation, outlier detection & treatment using IQR, and correlation analysis on the Titanic dataset.


## 1. Import Libraries & Load Data
We import `pandas`, `numpy`, `matplotlib.pyplot`, and `seaborn` to inspect the raw data.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df_raw = pd.read_csv('../data/raw_dataset.csv')
print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()


## 2. Data Profiling & Structural Inspection
Inspect column data types, non-null counts, and summary statistics to identify missing values and duplicate rows.


In [1]:
# General info & data types
df_raw.info()

# Summary statistics for numerical columns
df_raw.describe()

# Check for duplicates
num_duplicates = df_raw.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")


## 3. Mini-Task 1: Missing Value Profiling & Treatment
Calculate the exact percentage of missing values per column and apply domain-appropriate imputation.


In [1]:
# Missing value analysis
missing_counts = df_raw.isnull().sum()
missing_percent = (missing_counts / len(df_raw)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_counts, 'Percentage (%)': missing_percent})
print(missing_df[missing_df['Missing_Count'] > 0])

# Cleaning Step 1: Remove duplicate rows
df_cleaned = df_raw.drop_duplicates().reset_index(drop=True)

# Cleaning Step 2: Impute missing numerical 'Age' with median
age_median = df_cleaned['Age'].median()
df_cleaned['Age'] = df_cleaned['Age'].fillna(age_median)
print(f"Imputed Age missing values with median: {age_median}")

# Cleaning Step 3: Impute missing categorical 'Embarked' with mode
embarked_mode = df_cleaned['Embarked'].mode()[0]
df_cleaned['Embarked'] = df_cleaned['Embarked'].fillna(embarked_mode)
print(f"Imputed Embarked missing values with mode: '{embarked_mode}'")

print("\nRemaining missing values:")
print(df_cleaned.isnull().sum())


## 4. Mini-Task 2: Outlier Detection & Treatment via IQR
We visualize numerical distributions with box plots and apply the Interquartile Range (IQR) method to cap extreme values in `Fare`.


In [1]:
# Boxplot to inspect outliers in Fare
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.boxplot(y=df_cleaned['Fare'], color='#e74c3c')
plt.title('Fare Distribution Before IQR Treatment')

# Compute IQR for Fare
Q1 = df_cleaned['Fare'].quantile(0.25)
Q3 = df_cleaned['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"Fare Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"Outlier Thresholds -> Lower: {lower_bound:.2f}, Upper: {upper_bound:.2f}")

# Capping outliers to upper bound
df_cleaned['Fare'] = np.clip(df_cleaned['Fare'], lower_bound, upper_bound)

plt.subplot(1, 2, 2)
sns.boxplot(y=df_cleaned['Fare'], color='#2ecc71')
plt.title('Fare Distribution After IQR Capping')
plt.tight_layout()
plt.show()


## 5. Mini-Task 3: Correlation Analysis
We generate a Seaborn correlation heatmap to analyze relationships among numerical variables.


In [1]:
# Correlation matrix
numeric_df = df_cleaned.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.show()


### Key Correlation Insights:
1. **Passenger Class vs. Fare (`Pclass` & `Fare`)**: Demonstrates a moderate negative correlation (-0.56), confirming higher passenger classes (1st class) paid significantly higher ticket fares.
2. **Passenger Class vs. Survival (`Pclass` & `Survived`)**: Negative correlation indicates higher survival probability among passengers in higher classes with prioritized lifeboat access.
3. **Family Accompaniment (`SibSp` & `Parch`)**: Positive correlation (0.42) showing that passengers traveling with siblings or spouses were also likely to travel with parents or children.


## 6. Export Cleaned Dataset
Export the processed dataframe to `data/cleaned_dataset.csv`.


In [1]:
# Save cleaned dataset
df_cleaned.to_csv('../data/cleaned_dataset.csv', index=False)
print("Successfully exported cleaned dataset to data/cleaned_dataset.csv!")
